# Let's go

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statistics import mean, stdev
from scipy.stats import norm

In [ ]:
rcupd = {
    'figure.figsize': (5, 4),
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': 'cm',
    'font.size': 12,
}
plt.rcParams.update(rcupd)

In [ ]:
data_files = [
    '2025-11-04/RERTR5_V6018G.csv',
    '2025-11-04/RERTR12_L1P755.csv',
]

# Convenience functions

In [ ]:
def lin_coef_cept(mod):
    print(
        ' coeffs: ',
        mod.coef_, '\n',
        'intercept: ',
        mod.intercept_
    )

In [ ]:
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

def mod_metrics(mod, X_test, y_test):
    y_pred = mod.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    print(
        ' R2: ', r2, '\n',
        'RMSE: ', rmse, '\n',
        'MAE: ', mae
    )

In [ ]:
def pred_vs_actual(mod, X_test, y_test, tt):
    y_pred = mod.predict(X_test)

    plt.figure(figsize=(5,4))
    plt.rcParams.update({'font.size': 16})

    plt.scatter(y_test, y_pred, s=15)

    minv = int(min(min(y_test), min(y_pred)))
    maxv = int(max(max(y_test), max(y_pred)))
    val = list(range(minv, maxv))
    
    plt.plot(val, val, color='k', ls='--', label='y=x')

    plt.title(tt)
    plt.xlabel(r'Test data (swelling \%)')
    plt.ylabel(r'Surrogate pred. (swelling \%)')
    plt.legend()
    plt.show()

# Load

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [ ]:
def load_data(fileName):
    jar = pd.read_csv(fileName)

    col_names = jar.columns[1:-2]

    X = jar.iloc[:, 1:-2].to_numpy()
    y = jar.iloc[:, -2].to_numpy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=19
    )

    xscaler = MinMaxScaler()
    xscaler.fit(X_train)

    X_train = xscaler.transform(X_train)
    X_test = xscaler.transform(X_test)
    
    return X_train, X_test, y_train, y_test, xscaler, col_names

In [ ]:
X_train, X_test, y_train, y_test, xscaler, col_names = load_data(data_files[1])

# NN

In [ ]:
import pickle
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn import linear_model

In [ ]:
with open('nn_surrogates.pkl', 'rb') as f:
    regs = pickle.load(f)

In [ ]:
mod_metrics(regs[1], X_test, y_test)
pred_vs_actual(regs[1], X_test, y_test, 'Surrogate')

# Target

In [ ]:
def proposal_dist(X, sig):
    ret = []
    
    for el in X:
        prop = np.random.normal(el, sig)
        ret.append(prop)

    assert len(X) == len(ret)
    return ret

In [ ]:
def mcmc_sampler(num_param, initial_state, proposal_sig,
                 surrogates, target_fn, num_samples):
    samples = [initial_state]
    accepted = 0

    for ii in range(num_samples):
        current_state = samples[-1]
        proposed_state = proposal_dist(current_state, proposal_sig)

        valid = True
        for xx in proposed_state:
            if xx < 0 or xx > 1:
                valid = False
                break

        currs = [sur.predict([[*current_state]])[0] for sur in surrogates]
        props = [sur.predict([[*proposed_state]])[0] for sur in surrogates]
        fs_curr = mean(currs)
        fs_prop = mean(props)

        acceptance_ratio = target_fn.pdf(fs_prop) / target_fn.pdf(fs_curr)
        
        if valid and np.random.rand() < acceptance_ratio:
            current_state = proposed_state
            accepted += 1

        samples.append(current_state)

    print(f"Acceptance rate: {accepted / num_samples}")
    return np.array(samples)

In [ ]:
def swelling_perc(fd):
    return 3.83e-43 * fd**2 + 4.54e-21 * fd

In [ ]:
#target = norm(swelling_perc(5.34e21), 1.25)
target = norm(32, 2.64)

In [ ]:
hey1 = mcmc_sampler(
    9,
    np.random.rand(9),
    0.07,
    regs[1:],
    target,
    100000
)

In [ ]:
hey2 = mcmc_sampler(
    9,
    np.random.rand(9),
    0.07,
    regs[1:],
    target,
    100000
)

# Trace/Hist

In [ ]:
old1 = xscaler.inverse_transform(hey1)
old2 = xscaler.inverse_transform(hey2)

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(10, 10))

for i, ax in enumerate(axes.flatten()):
    cdat1 = old1[:,i]
    cavg1 = np.cumsum(cdat1) / np.arange(1, len(cdat1)+1)
    ax.plot(cdat1, lw=0.1, alpha=0.7, zorder=1)
    ax.plot(cavg1, c='k', zorder=2)
    
    cdat2 = old2[:,i]
    cavg2 = np.cumsum(cdat2) / np.arange(1, len(cdat2)+1)
    ax.plot(cdat2, ls='--', lw=0.1, alpha=0.7, zorder=1)
    ax.plot(cavg2, c='r', zorder=2)
    
    ax.set_xlabel(col_names[i])
    #ax.set_ylim([0, 1])

#fig.delaxes(axes[1,3])
fig.supylabel('Parameter values')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(10, 10))

for i, ax in enumerate(axes.flatten()):
    sns.histplot(old1[:,i], ax=ax, stat='density', kde=True)
    sns.histplot(old2[:,i], ax=ax, stat='density', kde=True)
    ax.set_xlabel(col_names[i])
    ax.set_ylabel('')
    #ax.set_xlim([0, 1])

#fig.delaxes(axes[1,3])
fig.supylabel('Density')
plt.tight_layout()
plt.show()

# Data thinning

In [ ]:
chey = np.concatenate((hey1[::100], hey2[::100]))

In [ ]:
orig = []
res = []
for i in range(2000):
    # pesky bug was here
    ress = [reg.predict([chey[-i]])[0] for reg in regs[1:]]
    res.append(mean(ress))

x = np.linspace(20, 45, 100)
y = target.pdf(x)
plt.plot(x, y, 'r', label='Obs. with noise')
plt.fill_between(x, y, color='r', alpha=0.5)

sns.histplot(res, kde=False, binwidth=1,
             ec='k', stat='density', label='Forward propagation')

plt.xlim([20, 45])
plt.xlabel(r'Fuel Swelling (\%)')
plt.legend(fontsize='x-small', loc='upper right')
plt.show()

In [ ]:
assert 2 == 3

# Posterior correlation

In [ ]:
cold = np.concatenate((old1[::100], old2[::100]))

In [ ]:
oldd = pd.DataFrame(cold, columns=col_names)

plt.figure(figsize=(10, 10))
sns.pairplot(
    pd.DataFrame(oldd),
    diag_kind='hist',
    diag_kws=dict(kde=True),
    plot_kws=dict(levels=6, bw_adjust=2.0, cmap='viridis'),
    kind='kde',
    corner=True
)

plt.show()

# Save

In [ ]:
oldd.to_csv('mcmc_samples.csv', index=False)